### Data Loading and Preprocessing 

In [ ]:
import numpy as np
import pandas as pd
from sklearn.preprocessing import StandardScaler, PolynomialFeatures
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error

# --- Load data ---
train_data = pd.read_excel("train_dataset_all_features.xlsx")
test_data = pd.read_excel("test_dataset_all_features.xlsx")
                            
print(f"Train data shape: {train_data.shape}")
print(f"Test data shape: {test_data.shape}")

features = ['C', 'Si', 'Mn', 'P', 'S', 'Ni', 'Cr', 'Mo', 'Cu','Al', 'N', 'N Temp', 'N Time', 'T Temp', 'T Time', 'Test Stress',	'Test Temp']
X_train = train_data[features]
y_train = train_data['Lg(CL)']  

X_test = test_data[features] 
y_test = test_data['Lg(CL)']    

print(f"X_train shape: {X_train.shape}")
print(f"y_train shape: {y_train.shape}")
print(f"X_test shape: {X_test.shape}") 
print(f"y_test shape: {y_test.shape}")

# Standardization 
s = StandardScaler()
X_train_s = s.fit_transform(X_train)  
X_test_s = s.transform(X_test)

print("✅ Data loaded and preprocessed successfully!")
print(f"X_train_s shape: {X_train_s.shape}")
print(f"X_test_s shape: {X_test_s.shape}")

In [ ]:
# ✅ ADD THIS LINE to see full parameters
pd.set_option('display.max_colwidth', 200)

### Linear Regression

In [ ]:
from sklearn.linear_model import LinearRegression
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.model_selection import cross_val_predict, KFold
from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error
import numpy as np
import pandas as pd

# ============================================================
# 🔄 LINEAR REGRESSION PIPELINE (LOG TARGET, LOG-SCALE METRICS)
# ============================================================
print("🔄 TRAINING LINEAR REGRESSION ON LOG TARGET (Evaluating in LOG SCALE)...")

pipeline_lr = Pipeline([
    ('scaler', StandardScaler()),
    ('lr', LinearRegression())
])

kf = KFold(n_splits=5, shuffle=True, random_state=42)

# ------------------------------------------------------------
# TRAIN MODEL
# ------------------------------------------------------------
pipeline_lr.fit(X_train, y_train)

y_pred_log_train_lr = pipeline_lr.predict(X_train)
y_pred_log_test_lr = pipeline_lr.predict(X_test)

# ------------------------------------------------------------
# METRICS FUNCTION (LOG SCALE)
# ------------------------------------------------------------
def compute_metrics(y_true, y_pred):
    r2_lr = r2_score(y_true, y_pred)
    mse_lr = mean_squared_error(y_true, y_pred)
    mae_lr = mean_absolute_error(y_true, y_pred)
    return r2_lr, mse_lr, mae_lr

r2_lr_train, mse_lr_train, mae_lr_train = compute_metrics(y_train, y_pred_log_train_lr)
r2_lr_test, mse_lr_test, mae_lr_test = compute_metrics(y_test, y_pred_log_test_lr)

# ------------------------------------------------------------
# CROSS-VALIDATION R² (LOG SCALE)
# ------------------------------------------------------------
y_cv_pred_log_lr = cross_val_predict(pipeline_lr, X_train, y_train, cv=kf)
cv_r2_lr = r2_score(y_train, y_cv_pred_log_lr)

# ------------------------------------------------------------
# DISPLAY RESULTS
# ------------------------------------------------------------
print("\n📊 LINEAR REGRESSION PERFORMANCE (LOG SCALE)")
print("="*60)
print(f"Training R² (log): {r2_lr_train:.4f}")
print(f"Testing  R² (log): {r2_lr_test:.4f}")
print(f"Cross-Validation R² (log): {cv_r2_lr:.4f}\n")

print(f"Training MSE (log): {mse_lr_train:.4f}")
print(f"Testing  MSE (log): {mse_lr_test:.4f}")
print(f"Training MAE (log): {mae_lr_train:.4f}")
print(f"Testing  MAE (log): {mae_lr_test:.4f}")

print("\nR² Difference (Train - Test):", round(r2_lr_train - r2_lr_test, 4))
print("Overfitting Indicator:", "⚠️ HIGH (Overfitting)" if (r2_lr_train - r2_lr_test) > 0.2 else "✅ GOOD")

# ------------------------------------------------------------
# SHOW FIRST 10 PREDICTIONS (LOG SCALE and Actual Scale)
# ------------------------------------------------------------

y_pred_train_lr = np.exp(y_pred_log_train_lr)
y_pred_test_lr = np.exp(y_pred_log_test_lr)
y_train_actual = np.exp(y_train)
y_test_actual = np.exp(y_test)

# Log-scale:

print("\nFirst 10 Predictions - Test Set (LOG SCALE)")
comparison_log_df_lr = pd.DataFrame({
    "Actual (log Rupture Time)": y_test[:10],
    "Predicted (log)": y_pred_log_test_lr[:10],
    "Error": y_test[:10] - y_pred_log_test_lr[:10]
}).round(4)
print(comparison_log_df_lr)

# Actual scale: 

print("\nFirst 10 Predictions - Test Set (Actual Scale)")
comparison_actual_df_lr = pd.DataFrame({
    "Actual (Rupture Time)": y_test_actual[:10],
    "Predicted": y_pred_test_lr[:10],
    "Error": y_test_actual[:10] - y_pred_test_lr[:10]
}).round(2)
print(comparison_actual_df_lr)

#### LR Plots

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
import pandas as pd

# Residuals
train_residuals_lr = y_train - y_pred_log_train_lr
test_residuals_lr = y_test - y_pred_log_test_lr

# -----------------------------
# SET PLOT STYLE
# -----------------------------
sns.set_style("whitegrid")
plt.rcParams.update({
    'figure.figsize': (7, 5),
    'axes.titlesize': 16,
    'axes.labelsize': 14,
    'xtick.labelsize': 12,
    'ytick.labelsize': 12,
    'lines.linewidth': 2,
    'lines.markersize': 6
})

# ============================================================
# 1️⃣ Predicted vs Actual (Log Scale)
# ============================================================
plt.figure()
sns.scatterplot(x=y_train, y=y_pred_log_train_lr, label='Train', color='steelblue')
sns.scatterplot(x=y_test, y=y_pred_log_test_lr, label='Test', color='orange')

# Perfect prediction line
lims = [min(y_train.min(), y_test.min()), max(y_train.max(), y_test.max())]
plt.plot(lims, lims, 'k--', alpha=0.7)
plt.xlabel("Actual log(Rupture Time)")
plt.ylabel("Predicted log(Rupture Time)")
plt.title("Linear Regression: Predicted vs Actual (Log Scale)")
plt.legend()
plt.tight_layout()
plt.show()

# ============================================================
# 2️⃣ Residuals Plot (Log Scale)
# ============================================================
plt.figure()
sns.scatterplot(x=y_train, y=train_residuals_lr, label='Train', color='steelblue')
sns.scatterplot(x=y_test, y=test_residuals_lr, label='Test', color='orange')
plt.axhline(0, color='k', linestyle='--', alpha=0.7)
plt.xlabel("Actual log(Rupture Time)")
plt.ylabel("Residuals")
plt.title("Linear Regression: Residuals vs Actual (Log Scale)")
plt.legend()
plt.tight_layout()
plt.show()

# ============================================================
# 3️⃣ Histogram of Errors (Log Scale)
# ============================================================
plt.figure()
sns.histplot(train_residuals_lr, color='steelblue', label='Train', kde=True, bins=30)
sns.histplot(test_residuals_lr, color='orange', label='Test', kde=True, bins=30)
plt.xlabel("Residuals")
plt.ylabel("Count")
plt.title("Linear Regression: Histogram of Residuals (Log Scale)")
plt.legend()
plt.tight_layout()
plt.show()


### LR ±2 Scatter Band Analysis

In [ ]:
# ============================================================
# 🔬 ±2 SCATTER BAND ANALYSIS: LINEAR REGRESSION
# ============================================================
print("🔬 ANALYZING ±2 SCATTER BAND FOR LINEAR REGRESSION...")
print("="*60)

# Calculate scatter ratios (Predicted/Actual)
scatter_ratio_lr = y_pred_test_lr / y_test_actual

# Scatter band metrics (FIXED variable names - no ± symbol)
within_2x_lr = np.mean((scatter_ratio_lr >= 0.5) & (scatter_ratio_lr <= 2.0)) * 100
within_1_5x_lr = np.mean((scatter_ratio_lr >= 0.67) & (scatter_ratio_lr <= 1.5)) * 100
within_3x_lr = np.mean((scatter_ratio_lr >= 0.33) & (scatter_ratio_lr <= 3.0)) * 100

# Conservative vs Optimistic predictions
conservative_lr = np.mean(scatter_ratio_lr < 1.0) * 100  # Under-predictions
optimistic_lr = np.mean(scatter_ratio_lr > 1.0) * 100    # Over-predictions

print(f"📊 LINEAR REGRESSION SCATTER BAND ANALYSIS:")
print(f"   ±2 Scatter Band:  {within_2x_lr:.1f}% of predictions")
print(f"   ±1.5 Scatter Band: {within_1_5x_lr:.1f}% of predictions") 
print(f"   ±3 Scatter Band:  {within_3x_lr:.1f}% of predictions")
print(f"   Conservative predictions (<1.0): {conservative_lr:.1f}%")
print(f"   Optimistic predictions (>1.0): {optimistic_lr:.1f}%")

# ============================================================
# SCATTER BAND PLOT
# ============================================================
plt.figure(figsize=(10, 8))

# Scatter plot
plt.scatter(y_test_actual, y_pred_test_lr, alpha=0.7, color='blue', s=50, 
           label=f'LR Predictions ({within_2x_lr:.1f}% within ±2 band)')

# Reference lines
max_val = max(y_test_actual.max(), y_pred_test_lr.max())
min_val = min(y_test_actual.min(), y_pred_test_lr.min())

# Perfect prediction line
plt.plot([min_val, max_val], [min_val, max_val], 'k-', label='Perfect Prediction', linewidth=2)

# ±2 scatter band lines
plt.plot([min_val, max_val], [0.5*min_val, 0.5*max_val], 'r--', label='±2 Scatter Band', linewidth=1.5)
plt.plot([min_val, max_val], [2*min_val, 2*max_val], 'r--', linewidth=1.5)

plt.xscale('log')
plt.yscale('log')
plt.xlabel('Actual Rupture Time (hours)')
plt.ylabel('Predicted Rupture Time (hours)')
plt.title('Linear Regression: ±2 Scatter Band Analysis\n(Actual Scale)')
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

# ============================================================
# SCATTER RATIO DISTRIBUTION (IMPROVED VERSION)
# ============================================================
plt.figure(figsize=(10, 6))

# Use consistent binning and add statistics
n, bins, patches = plt.hist(scatter_ratio_lr, bins=50, alpha=0.7, color='blue', 
                           edgecolor='black', density=False)

plt.axvline(1.0, color='red', linestyle='--', linewidth=2, label='Perfect Prediction (1.0)')
plt.axvline(0.5, color='orange', linestyle='--', linewidth=1.5, label='±2 Band Limits')
plt.axvline(2.0, color='orange', linestyle='--', linewidth=1.5)

# Add statistical annotations
median_ratio = np.median(scatter_ratio_lr)
mean_ratio = np.mean(scatter_ratio_lr)

plt.axvline(median_ratio, color='green', linestyle=':', linewidth=2, 
           label=f'Median Ratio: {median_ratio:.2f}')

plt.xlabel('Predicted / Actual Ratio')
plt.ylabel('Frequency')
plt.title(f'Linear Regression: Scatter Ratio Distribution\n'
          f'Mean: {mean_ratio:.2f}, Median: {median_ratio:.2f}, '
          f'Within ±2: {within_2x_lr:.1f}%')
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()

# Add text box with key statistics
textstr = f'''Key Statistics:
N = {len(scatter_ratio_lr)}
Mean = {mean_ratio:.2f}
Median = {median_ratio:.2f}
Std = {np.std(scatter_ratio_lr):.2f}
Within ±2: {within_2x_lr:.1f}%'''
props = dict(boxstyle='round', facecolor='wheat', alpha=0.8)
plt.gca().text(0.95, 0.95, textstr, transform=plt.gca().transAxes, 
               verticalalignment='top', horizontalalignment='right', bbox=props)

plt.show()

### SVR

In [ ]:
from sklearn.svm import SVR
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.model_selection import GridSearchCV, KFold, cross_val_predict
from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error
import numpy as np
import pandas as pd

# ============================================================
# 🔄 SVR PIPELINE (LOG TARGET, LOG-SCALE METRICS)
# ============================================================
print("🔄 PERFORMING SVR HYPERPARAMETER TUNING ON LOG TARGET...")

# Pipeline for scaling + SVR
pipeline_svr = Pipeline([
    ('scaler', StandardScaler()),
    ('svr', SVR())
])

# Parameter grid
param_grid_svr = {
    'svr__C': [0.1, 1, 10, 100],
    'svr__gamma': ['scale', 'auto', 0.1, 0.01],
    'svr__kernel': ['rbf', 'linear']
}

kf = KFold(n_splits=5, shuffle=True, random_state=42)

grid_search_svr = GridSearchCV(
    pipeline_svr,
    param_grid_svr,
    cv=kf,
    scoring='r2',
    n_jobs=-1,
    verbose=1,
    refit=True
)

# ------------------------------------------------------------
# FIT GRID SEARCH
# ------------------------------------------------------------
grid_search_svr.fit(X_train, y_train)
best_svr_pipeline = grid_search_svr.best_estimator_

print(f"🎯 Best SVR Parameters: {grid_search_svr.best_params_}")
print(f"🎯 Best CV R² (log scale): {grid_search_svr.best_score_:.4f}")

# SHOW TOP PARAMETER COMBINATIONS (This is optional) 
# ------------------------------------------------------------
results_df_svr = pd.DataFrame(grid_search_svr.cv_results_)
print(f"\n🔍 Top 5 SVR parameter combinations:")
print(results_df_svr[['params', 'mean_test_score']].sort_values('mean_test_score', ascending=False).head(5))
# ------------------------------------------------------------

# ------------------------------------------------------------
# PREDICTIONS (LOG SCALE)
# ------------------------------------------------------------
y_pred_log_train_svr = best_svr_pipeline.predict(X_train)
y_pred_log_test_svr = best_svr_pipeline.predict(X_test)

# ------------------------------------------------------------
# METRICS FUNCTION (LOG SCALE)
# ------------------------------------------------------------
def compute_metrics(y_true, y_pred):
    r2_svr = r2_score(y_true, y_pred)
    mse_svr = mean_squared_error(y_true, y_pred)
    mae_svr = mean_absolute_error(y_true, y_pred)
    return r2_svr, mse_svr, mae_svr

# Training & Testing metrics (log scale)
r2_svr_train, mse_svr_train, mae_svr_train = compute_metrics(y_train, y_pred_log_train_svr)
r2_svr_test, mse_svr_test, mae_svr_test = compute_metrics(y_test, y_pred_log_test_svr)

# ------------------------------------------------------------
# CROSS-VALIDATION R² (LOG SCALE)
# ------------------------------------------------------------
y_cv_pred_log_svr = cross_val_predict(best_svr_pipeline, X_train, y_train, cv=kf)
cv_r2_svr = r2_score(y_train, y_cv_pred_log_svr)

# ------------------------------------------------------------
# DISPLAY RESULTS (LOG SCALE)
# ------------------------------------------------------------
print("\n📊 SVR PERFORMANCE (LOG SCALE)")
print("="*60)
print(f"Training R² (log): {r2_svr_train:.4f}")
print(f"Testing  R² (log): {r2_svr_test:.4f}")
print(f"Cross-Validation R² (log): {cv_r2_svr:.4f}\n")

print(f"Training MSE (log): {mse_svr_train:.4f}")
print(f"Testing  MSE (log): {mse_svr_test:.4f}")
print(f"Training MAE (log): {mae_svr_train:.4f}")
print(f"Testing  MAE (log): {mae_svr_test:.4f}")

print("\nR² Difference (Train - Test):", round(r2_svr_train - r2_svr_test, 4))
print("Overfitting Indicator:", "⚠️ HIGH (Overfitting)" if (r2_svr_train - r2_svr_test) > 0.2 else "✅ GOOD")

# ------------------------------------------------------------
# PREDICTIONS IN BOTH LOG AND ACTUAL SCALE
# ------------------------------------------------------------
y_pred_train_svr = np.exp(y_pred_log_train_svr)
y_pred_test_svr = np.exp(y_pred_log_test_svr)
y_train_actual = np.exp(y_train)
y_test_actual = np.exp(y_test)

# First 10 predictions in LOG SCALE
print("\nFirst 10 Predictions - Test Set (LOG SCALE)")
comparison_log_df_svr = pd.DataFrame({
    "Actual (log Rupture Time)": y_test[:10],
    "Predicted (log)": y_pred_log_test_svr[:10],
    "Error": y_test[:10] - y_pred_log_test_svr[:10]
}).round(4)
print(comparison_log_df_svr)

# First 10 predictions in ACTUAL SCALE
print("\nFirst 10 Predictions - Test Set (ACTUAL SCALE)")
comparison_actual_df_svr = pd.DataFrame({
    "Actual (Rupture Time)": y_test_actual[:10],
    "Predicted": y_pred_test_svr[:10],
    "Error": y_test_actual[:10] - y_pred_test_svr[:10]
}).round(2)
print(comparison_actual_df_svr)

### SVR Plots

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# -----------------------------
# Residuals
# -----------------------------
train_residuals_svr = y_train - y_pred_log_train_svr
test_residuals_svr = y_test - y_pred_log_test_svr

# ============================================================
# 1️⃣ Predicted vs Actual (Log Scale)
# ============================================================
plt.figure(figsize=(7,6))
sns.scatterplot(x=y_train, y=y_pred_log_train_svr, label='Train', color='steelblue', s=60)
sns.scatterplot(x=y_test, y=y_pred_log_test_svr, label='Test', color='orange', s=60)
plt.plot([y_train.min(), y_train.max()], [y_train.min(), y_train.max()], 'k--', alpha=0.7)
plt.xlabel("Actual log(Rupture Time)")
plt.ylabel("Predicted log(Rupture Time)")
plt.title("SVR: Predicted vs Actual (Log Scale)")
plt.legend()
plt.tight_layout()
plt.show()

# ============================================================
# 2️⃣ Residuals vs Actual (Log Scale)
# ============================================================
plt.figure(figsize=(7,6))
sns.scatterplot(x=y_train, y=train_residuals_svr, label='Train', color='steelblue', s=60)
sns.scatterplot(x=y_test, y=test_residuals_svr, label='Test', color='orange', s=60)
plt.axhline(0, color='k', linestyle='--', alpha=0.7)
plt.xlabel("Actual log(Rupture Time)")
plt.ylabel("Residuals")
plt.title("SVR: Residuals vs Actual (Log Scale)")
plt.legend()
plt.tight_layout()
plt.show()

# ============================================================
# 3️⃣ Histogram of Residuals (Log Scale)
# ============================================================
plt.figure(figsize=(7,6))
sns.histplot(train_residuals_svr, color='steelblue', label='Train', kde=True, bins=30)
sns.histplot(test_residuals_svr, color='orange', label='Test', kde=True, bins=30)
plt.xlabel("Residuals")
plt.ylabel("Count") 
plt.title("SVR: Histogram of Residuals (Log Scale)")
plt.legend()
plt.tight_layout()
plt.show()


### SVR ±2 Scatter Band Analysis

In [ ]:
# ============================================================
# 🔬 ±2 SCATTER BAND ANALYSIS: SVR
# ============================================================
print("🔬 ANALYZING ±2 SCATTER BAND FOR SVR...")
print("="*60)

# Calculate scatter ratios (Predicted/Actual)
scatter_ratio_svr = y_pred_test_svr / y_test_actual

# Scatter band metrics (FIXED variable names - no ± symbol)
within_2x_svr = np.mean((scatter_ratio_svr >= 0.5) & (scatter_ratio_svr <= 2.0)) * 100
within_1_5x_svr = np.mean((scatter_ratio_svr >= 0.67) & (scatter_ratio_svr <= 1.5)) * 100
within_3x_svr = np.mean((scatter_ratio_svr >= 0.33) & (scatter_ratio_svr <= 3.0)) * 100

# Conservative vs Optimistic predictions
conservative_svr = np.mean(scatter_ratio_svr < 1.0) * 100  # Under-predictions
optimistic_svr = np.mean(scatter_ratio_svr > 1.0) * 100    # Over-predictions

print(f"📊 SVR SCATTER BAND ANALYSIS:")
print(f"   ±2 Scatter Band:  {within_2x_svr:.1f}% of predictions")
print(f"   ±1.5 Scatter Band: {within_1_5x_svr:.1f}% of predictions") 
print(f"   ±3 Scatter Band:  {within_3x_svr:.1f}% of predictions")
print(f"   Conservative predictions (<1.0): {conservative_svr:.1f}%")
print(f"   Optimistic predictions (>1.0): {optimistic_svr:.1f}%")

# ============================================================
# SCATTER BAND PLOT
# ============================================================
plt.figure(figsize=(10, 8))

# Scatter plot
plt.scatter(y_test_actual, y_pred_test_svr, alpha=0.7, color='blue', s=50, 
           label=f'SVR Predictions ({within_2x_svr:.1f}% within ±2 band)')

# Reference lines
max_val = max(y_test_actual.max(), y_pred_test_svr.max())
min_val = min(y_test_actual.min(), y_pred_test_svr.min())

# Perfect prediction line
plt.plot([min_val, max_val], [min_val, max_val], 'k-', label='Perfect Prediction', linewidth=2)

# ±2 scatter band lines
plt.plot([min_val, max_val], [0.5*min_val, 0.5*max_val], 'r--', label='±2 Scatter Band', linewidth=1.5)
plt.plot([min_val, max_val], [2*min_val, 2*max_val], 'r--', linewidth=1.5)

plt.xscale('log')
plt.yscale('log')
plt.xlabel('Actual Rupture Time (hours)')
plt.ylabel('Predicted Rupture Time (hours)')
plt.title('SVR: ±2 Scatter Band Analysis\n(Actual Scale)')
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

# ============================================================
# SCATTER RATIO DISTRIBUTION (IMPROVED VERSION)
# ============================================================
plt.figure(figsize=(10, 6))

# Use consistent binning and add statistics
n, bins, patches = plt.hist(scatter_ratio_svr, bins=50, alpha=0.7, color='blue', 
                           edgecolor='black', density=False)

plt.axvline(1.0, color='red', linestyle='--', linewidth=2, label='Perfect Prediction (1.0)')
plt.axvline(0.5, color='orange', linestyle='--', linewidth=1.5, label='±2 Band Limits')
plt.axvline(2.0, color='orange', linestyle='--', linewidth=1.5)

# Add statistical annotations
median_ratio = np.median(scatter_ratio_svr)
mean_ratio = np.mean(scatter_ratio_svr)

plt.axvline(median_ratio, color='green', linestyle=':', linewidth=2, 
           label=f'Median Ratio: {median_ratio:.2f}')

plt.xlabel('Predicted / Actual Ratio')
plt.ylabel('Frequency')
plt.title(f'SVR: Scatter Ratio Distribution\n'
          f'Mean: {mean_ratio:.2f}, Median: {median_ratio:.2f}, '
          f'Within ±2: {within_2x_svr:.1f}%')
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()

# Add text box with key statistics
textstr = f'''Key Statistics:
N = {len(scatter_ratio_svr)}
Mean = {mean_ratio:.2f}
Median = {median_ratio:.2f}
Std = {np.std(scatter_ratio_svr):.2f}
Within ±2: {within_2x_svr:.1f}%'''
props = dict(boxstyle='round', facecolor='wheat', alpha=0.8)
plt.gca().text(0.95, 0.95, textstr, transform=plt.gca().transAxes, 
               verticalalignment='top', horizontalalignment='right', bbox=props)

plt.show()

### Random Forest

In [ ]:
from sklearn.ensemble import RandomForestRegressor
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.model_selection import GridSearchCV, KFold, cross_val_predict
from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error
import numpy as np
import pandas as pd

# ============================================================
# 🔄 RANDOM FOREST PIPELINE (LOG TARGET, LOG-SCALE METRICS)
# ============================================================
print("🔄 Performing Random Forest hyperparameter tuning on LOG target...")

pipeline_rf = Pipeline([
    ('scaler', StandardScaler()),
    ('rf', RandomForestRegressor(random_state=42))
])

param_grid_rf = {
    'rf__n_estimators': [100, 200, 500],
    'rf__max_depth': [None, 5, 10, 20],
    'rf__min_samples_split': [2, 5, 10],
    'rf__min_samples_leaf': [1, 2, 4],
    'rf__max_features': ['sqrt', 'log2', None]
}

kf = KFold(n_splits=5, shuffle=True, random_state=42)

grid_search_rf = GridSearchCV(
    pipeline_rf,
    param_grid_rf,
    cv=kf,
    scoring='r2',
    n_jobs=-1,
    verbose=1,
    refit=True
)

# ------------------------------------------------------------
# FIT GRID SEARCH
# ------------------------------------------------------------
grid_search_rf.fit(X_train, y_train)
best_rf_pipeline = grid_search_rf.best_estimator_

print(f"🎯 Best RF Parameters: {grid_search_rf.best_params_}")
print(f"🎯 Best CV R² (log scale): {grid_search_rf.best_score_:.4f}")


# SHOW TOP PARAMETER COMBINATIONS (This is optional) 
# ------------------------------------------------------------
results_df_rf = pd.DataFrame(grid_search_rf.cv_results_)
print(f"\n🔍 Top 5 RF parameter combinations:")
print(results_df_rf[['params', 'mean_test_score']].sort_values('mean_test_score', ascending=False).head(5))
# ------------------------------------------------------------

# ------------------------------------------------------------
# PREDICTIONS (LOG SCALE)
# ------------------------------------------------------------
y_pred_log_train_rf = best_rf_pipeline.predict(X_train)
y_pred_log_test_rf = best_rf_pipeline.predict(X_test)

# ------------------------------------------------------------
# METRICS FUNCTION (LOG SCALE)
# ------------------------------------------------------------
def compute_metrics(y_true, y_pred):
    r2_rf = r2_score(y_true, y_pred)
    mse_rf = mean_squared_error(y_true, y_pred)
    mae_rf = mean_absolute_error(y_true, y_pred)
    return r2_rf, mse_rf, mae_rf

# Training & Testing metrics (log scale)
r2_rf_train, mse_rf_train, mae_rf_train = compute_metrics(y_train, y_pred_log_train_rf)
r2_rf_test, mse_rf_test, mae_rf_test = compute_metrics(y_test, y_pred_log_test_rf)

# ------------------------------------------------------------
# CROSS-VALIDATION R² (LOG SCALE)
# ------------------------------------------------------------
y_cv_pred_log_rf = cross_val_predict(best_rf_pipeline, X_train, y_train, cv=kf)
cv_r2_rf = r2_score(y_train, y_cv_pred_log_rf)

# ------------------------------------------------------------
# DISPLAY RESULTS (LOG SCALE)
# ------------------------------------------------------------
print("\n📊 RANDOM FOREST PERFORMANCE (LOG SCALE)")
print("="*60)
print(f"Training R² (log): {r2_rf_train:.4f}")
print(f"Testing  R² (log): {r2_rf_test:.4f}")
print(f"Cross-Validation R² (log): {cv_r2_rf:.4f}\n")

print(f"Training MSE (log): {mse_rf_train:.4f}")
print(f"Testing  MSE (log): {mse_rf_test:.4f}")
print(f"Training MAE (log): {mae_rf_train:.4f}")
print(f"Testing  MAE (log): {mae_rf_test:.4f}")

print("\nR² Difference (Train - Test):", round(r2_rf_train - r2_rf_test, 4))
print("Overfitting Indicator:", "⚠️ HIGH (Overfitting)" if (r2_rf_train - r2_rf_test) > 0.2 else "✅ GOOD")

# ------------------------------------------------------------
# PREDICTIONS IN BOTH LOG AND ACTUAL SCALE

y_pred_train_rf = np.exp(y_pred_log_train_rf)
y_train_actual = np.exp(y_train)
y_pred_test_rf = np.exp(y_pred_log_test_rf)
y_test_actual = np.exp(y_test)

# First 10 predictions in LOG SCALE
print("\nFirst 10 Predictions - Test Set (LOG SCALE)")
comparison_log_df_rf = pd.DataFrame({
    "Actual (log Rupture Time)": y_test[:10],
    "Predicted (log)": y_pred_log_test_rf[:10],
    "Error": y_test[:10] - y_pred_log_test_rf[:10]
}).round(4)
print(comparison_log_df_rf)

# First 10 predictions in ACTUAL SCALE
print("\nFirst 10 Predictions - Test Set (ACTUAL SCALE)")
comparison_actual_df_rf = pd.DataFrame({
    "Actual (Rupture Time)": y_test_actual[:10],
    "Predicted": y_pred_test_rf[:10],
    "Error": y_test_actual[:10] - y_pred_test_rf[:10]
}).round(2)
print(comparison_actual_df_rf)


### RF Plots

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
import pandas as pd

# Residuals
train_residuals_rf = y_train - y_pred_log_train_rf
test_residuals_rf = y_test - y_pred_log_test_rf

# -----------------------------
# SET PLOT STYLE
# -----------------------------
sns.set_style("whitegrid")
plt.rcParams.update({
    'figure.figsize': (7, 5),
    'axes.titlesize': 16,
    'axes.labelsize': 14,
    'xtick.labelsize': 12,
    'ytick.labelsize': 12,
    'lines.linewidth': 2,
    'lines.markersize': 6
})

# ============================================================
# 1️⃣ Predicted vs Actual (Log Scale)
# ============================================================
plt.figure()
sns.scatterplot(x=y_train, y=y_pred_log_train_rf, label='Train', color='steelblue')
sns.scatterplot(x=y_test, y=y_pred_log_test_rf, label='Test', color='orange')

# Perfect prediction line
lims = [min(y_train.min(), y_test.min()), max(y_train.max(), y_test.max())]
plt.plot(lims, lims, 'k--', alpha=0.7)
plt.xlabel("Actual log(Rupture Time)")
plt.ylabel("Predicted log(Rupture Time)")
plt.title("RF: Predicted vs Actual (Log Scale)")
plt.legend()
plt.tight_layout()
plt.show()

# ============================================================
# 2️⃣ Residuals Plot (Log Scale)
# ============================================================
plt.figure()
sns.scatterplot(x=y_train, y=train_residuals_rf, label='Train', color='steelblue')
sns.scatterplot(x=y_test, y=test_residuals_rf, label='Test', color='orange')
plt.axhline(0, color='k', linestyle='--', alpha=0.7)
plt.xlabel("Actual log(Rupture Time)")
plt.ylabel("Residuals")
plt.title("RF: Residuals vs Actual (Log Scale)")
plt.legend()
plt.tight_layout()
plt.show()

# ============================================================
# 3️⃣ Histogram of Errors (Log Scale)
# ============================================================
plt.figure()
sns.histplot(train_residuals_rf, color='steelblue', label='Train', kde=True, bins=30)
sns.histplot(test_residuals_rf, color='orange', label='Test', kde=True, bins=30)
plt.xlabel("Residuals")
plt.ylabel("Count")
plt.title("RF: Histogram of Residuals (Log Scale)")
plt.legend()
plt.tight_layout()
plt.show()


### RF ±2 Scatter Band Analysis

In [ ]:
# ============================================================
# 🔬 ±2 SCATTER BAND ANALYSIS: RF
# ============================================================
print("🔬 ANALYZING ±2 SCATTER BAND FOR RF...")
print("="*60)

# Calculate scatter ratios (Predicted/Actual)
scatter_ratio_rf = y_pred_test_rf / y_test_actual

# Scatter band metrics (FIXED variable names - no ± symbol)
within_2x_rf = np.mean((scatter_ratio_rf >= 0.5) & (scatter_ratio_rf <= 2.0)) * 100
within_1_5x_rf = np.mean((scatter_ratio_rf >= 0.67) & (scatter_ratio_rf <= 1.5)) * 100
within_3x_rf = np.mean((scatter_ratio_rf >= 0.33) & (scatter_ratio_rf <= 3.0)) * 100

# Conservative vs Optimistic predictions
conservative_rf = np.mean(scatter_ratio_rf < 1.0) * 100  # Under-predictions
optimistic_rf = np.mean(scatter_ratio_rf > 1.0) * 100    # Over-predictions

print(f"📊 RF SCATTER BAND ANALYSIS:")
print(f"   ±2 Scatter Band:  {within_2x_rf:.1f}% of predictions")
print(f"   ±1.5 Scatter Band: {within_1_5x_rf:.1f}% of predictions") 
print(f"   ±3 Scatter Band:  {within_3x_rf:.1f}% of predictions")
print(f"   Conservative predictions (<1.0): {conservative_rf:.1f}%")
print(f"   Optimistic predictions (>1.0): {optimistic_rf:.1f}%")

# ============================================================
# SCATTER BAND PLOT
# ============================================================
plt.figure(figsize=(10, 8))

# Scatter plot
plt.scatter(y_test_actual, y_pred_test_rf, alpha=0.7, color='blue', s=50, 
           label=f'RF Predictions ({within_2x_rf:.1f}% within ±2 band)')

# Reference lines
max_val = max(y_test_actual.max(), y_pred_test_rf.max())
min_val = min(y_test_actual.min(), y_pred_test_rf.min())

# Perfect prediction line
plt.plot([min_val, max_val], [min_val, max_val], 'k-', label='Perfect Prediction', linewidth=2)

# ±2 scatter band lines
plt.plot([min_val, max_val], [0.5*min_val, 0.5*max_val], 'r--', label='±2 Scatter Band', linewidth=1.5)
plt.plot([min_val, max_val], [2*min_val, 2*max_val], 'r--', linewidth=1.5)

plt.xscale('log')
plt.yscale('log')
plt.xlabel('Actual Rupture Time (hours)')
plt.ylabel('Predicted Rupture Time (hours)')
plt.title('RF: ±2 Scatter Band Analysis\n(Actual Scale)')
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

# ============================================================
# SCATTER RATIO DISTRIBUTION (IMPROVED VERSION)
# ============================================================
plt.figure(figsize=(10, 6))

# Use consistent binning and add statistics
n, bins, patches = plt.hist(scatter_ratio_rf, bins=50, alpha=0.7, color='blue', 
                           edgecolor='black', density=False)

plt.axvline(1.0, color='red', linestyle='--', linewidth=2, label='Perfect Prediction (1.0)')
plt.axvline(0.5, color='orange', linestyle='--', linewidth=1.5, label='±2 Band Limits')
plt.axvline(2.0, color='orange', linestyle='--', linewidth=1.5)

# Add statistical annotations
median_ratio = np.median(scatter_ratio_rf)
mean_ratio = np.mean(scatter_ratio_rf)

plt.axvline(median_ratio, color='green', linestyle=':', linewidth=2, 
           label=f'Median Ratio: {median_ratio:.2f}')

plt.xlabel('Predicted / Actual Ratio')
plt.ylabel('Frequency')
plt.title(f'RF: Scatter Ratio Distribution\n'
          f'Mean: {mean_ratio:.2f}, Median: {median_ratio:.2f}, '
          f'Within ±2: {within_2x_rf:.1f}%')
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()

# Add text box with key statistics
textstr = f'''Key Statistics:
N = {len(scatter_ratio_rf)}
Mean = {mean_ratio:.2f}
Median = {median_ratio:.2f}
Std = {np.std(scatter_ratio_rf):.2f}
Within ±2: {within_2x_rf:.1f}%'''
props = dict(boxstyle='round', facecolor='wheat', alpha=0.8)
plt.gca().text(0.95, 0.95, textstr, transform=plt.gca().transAxes, 
               verticalalignment='top', horizontalalignment='right', bbox=props)

plt.show()

### GBR

In [ ]:
from sklearn.ensemble import GradientBoostingRegressor
from sklearn.model_selection import GridSearchCV, KFold, cross_val_predict
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error
import numpy as np
import pandas as pd

# ============================================================
# 🔄 GRADIENT BOOSTING REGRESSION PIPELINE (LOG TARGET, LOG-SCALE METRICS)
# ============================================================

print("🔄 Performing Gradient Boosting Regression hyperparameter tuning on LOG target...")

pipeline_gbr = Pipeline([
    ('scaler', StandardScaler()),
    ('gbr', GradientBoostingRegressor(random_state=42))
])

param_grid_gbr = {
    'gbr__n_estimators': [100, 200, 500],
    'gbr__learning_rate': [0.01, 0.05, 0.1],
    'gbr__max_depth': [2, 3, 4],
    'gbr__min_samples_split': [2, 5],
    'gbr__min_samples_leaf': [1, 2]
}

kf = KFold(n_splits=5, shuffle=True, random_state=42)

grid_search_gbr = GridSearchCV(
    pipeline_gbr,
    param_grid_gbr,
    cv=kf,
    scoring='r2',
    n_jobs=-1,
    verbose=1,
    refit=True
)

# ------------------------------------------------------------
# FIT GRID SEARCH
# ------------------------------------------------------------

grid_search_gbr.fit(X_train, y_train)
best_gbr_pipeline = grid_search_gbr.best_estimator_

print(f"🎯 Best GBR Parameters: {grid_search_gbr.best_params_}")
print(f"🎯 Best CV R² (log scale): {grid_search_gbr.best_score_:.4f}")

# SHOW TOP PARAMETER COMBINATIONS (This is optional) 
# ------------------------------------------------------------
results_df_gbr = pd.DataFrame(grid_search_gbr.cv_results_)
print(f"\n🔍 Top 5 GBR parameter combinations:")
print(results_df_gbr[['params', 'mean_test_score']].sort_values('mean_test_score', ascending=False).head(5))
# ------------------------------------------------------------

# ============================================================
# PREDICTIONS (LOG SCALE)
# ============================================================
y_pred_log_train_gbr = best_gbr_pipeline.predict(X_train)
y_pred_log_test_gbr = best_gbr_pipeline.predict(X_test)

# ============================================================
# METRICS FUNCTION (LOG SCALE)
# ============================================================

def compute_metrics(y_true, y_pred):
    r2_gbr = r2_score(y_true, y_pred)
    mse_gbr = mean_squared_error(y_true, y_pred)
    mae_gbr = mean_absolute_error(y_true, y_pred)
    return r2_gbr, mse_gbr, mae_gbr

# Training & Testing metrics (log scale)
r2_gbr_train, mse_gbr_train, mae_gbr_train = compute_metrics(y_train, y_pred_log_train_gbr)
r2_gbr_test, mse_gbr_test, mae_gbr_test = compute_metrics(y_test, y_pred_log_test_gbr)

# ------------------------------------------------------------
# CROSS-VALIDATION R² (LOG SCALE)
# ------------------------------------------------------------
y_cv_pred_log_gbr = cross_val_predict(best_gbr_pipeline, X_train, y_train, cv=kf)
cv_r2_gbr = r2_score(y_train, y_cv_pred_log_gbr)

# ============================================================
# DISPLAY RESULTS (LOG SCALE)
# ============================================================
print("\n📊 GRADIENT BOOSTING REGRESSOR PERFORMANCE (Log Scale)")
print("="*60)
print(f"Training R² (log): {r2_gbr_train:.4f}")
print(f"Testing  R² (log): {r2_gbr_test:.4f}")
print(f"Cross-Validation R² (log): {cv_r2_gbr:.4f}\n")

print(f"Training MSE (log): {mse_gbr_train:.4f}")
print(f"Testing  MSE (log): {mse_gbr_test:.4f}")
print(f"Training MAE (log): {mae_gbr_train:.4f}")
print(f"Testing  MAE (log): {mae_gbr_test:.4f}")

print("\nR² Difference (Train - Test):", round(r2_gbr_train - r2_gbr_test, 4))
print("Overfitting Indicator:", "⚠️ HIGH (Overfitting)" if (r2_gbr_train - r2_gbr_test) > 0.2 else "✅ GOOD")

# ------------------------------------------------------------
# PREDICTIONS IN BOTH LOG AND ACTUAL SCALE

y_pred_train_gbr = np.exp(y_pred_log_train_gbr)
y_pred_test_gbr = np.exp(y_pred_log_test_gbr)
y_train_actual = np.exp(y_train)
y_test_actual = np.exp(y_test)

# First 10 predictions in LOG SCALE
print("\nFirst 10 Predictions - Test Set (LOG SCALE)")
comparison_log_df_gbr = pd.DataFrame({
    "Actual (log Rupture Time)": y_test[:10],
    "Predicted (log)": y_pred_log_test_gbr[:10],
    "Error": y_test[:10] - y_pred_log_test_gbr[:10]
}).round(4)
print(comparison_log_df_gbr)

# First 10 predictions in ACTUAL SCALE
print("\nFirst 10 Predictions - Test Set (ACTUAL SCALE)")
comparison_actual_df_gbr = pd.DataFrame({
    "Actual (Rupture Time)": y_test_actual[:10],
    "Predicted": y_pred_test_gbr[:10],
    "Error": y_test_actual[:10] - y_pred_test_gbr[:10]
}).round(2)
print(comparison_actual_df_gbr)


### GBR Plots

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
import pandas as pd

# Residuals
train_residuals_gbr = y_train - y_pred_log_train_gbr
test_residuals_gbr = y_test - y_pred_log_test_gbr

# -----------------------------
# SET PLOT STYLE
# -----------------------------
sns.set_style("whitegrid")
plt.rcParams.update({
    'figure.figsize': (7, 5),
    'axes.titlesize': 16,
    'axes.labelsize': 14,
    'xtick.labelsize': 12,
    'ytick.labelsize': 12,
    'lines.linewidth': 2,
    'lines.markersize': 6
})

# ============================================================
# 1️⃣ Predicted vs Actual (Log Scale)
# ============================================================
plt.figure()
sns.scatterplot(x=y_train, y=y_pred_log_train_gbr, label='Train', color='steelblue')
sns.scatterplot(x=y_test, y=y_pred_log_test_gbr, label='Test', color='orange')

# Perfect prediction line
lims = [min(y_train.min(), y_test.min()), max(y_train.max(), y_test.max())]
plt.plot(lims, lims, 'k--', alpha=0.7)
plt.xlabel("Actual log(Rupture Time)")
plt.ylabel("Predicted log(Rupture Time)")
plt.title("GBR: Predicted vs Actual (Log Scale)")
plt.legend()
plt.tight_layout()
plt.show()

# ============================================================
# 2️⃣ Residuals Plot (Log Scale)
# ============================================================
plt.figure()
sns.scatterplot(x=y_train, y=train_residuals_gbr, label='Train', color='steelblue')
sns.scatterplot(x=y_test, y=test_residuals_gbr, label='Test', color='orange')
plt.axhline(0, color='k', linestyle='--', alpha=0.7)
plt.xlabel("Actual log(Rupture Time)")
plt.ylabel("Residuals")
plt.title("GBR: Residuals vs Actual (Log Scale)")
plt.legend()
plt.tight_layout()
plt.show()

# ============================================================
# 3️⃣ Histogram of Errors (Log Scale)
# ============================================================
plt.figure()
sns.histplot(train_residuals_gbr, color='steelblue', label='Train', kde=True, bins=30)
sns.histplot(test_residuals_gbr, color='orange', label='Test', kde=True, bins=30)
plt.xlabel("Residuals")
plt.ylabel("Count")
plt.title("GBR: Histogram of Residuals (Log Scale)")
plt.legend()
plt.tight_layout()
plt.show()


### GBR ±2 Scatter Band Analysis

In [ ]:
# ============================================================
# 🔬 ±2 SCATTER BAND ANALYSIS: GBR
# ============================================================
print("🔬 ANALYZING ±2 SCATTER BAND FOR GBR...")
print("="*60)

# Calculate scatter ratios (Predicted/Actual)
scatter_ratio_gbr = y_pred_test_gbr / y_test_actual

# Scatter band metrics (FIXED variable names - no ± symbol)
within_2x_gbr = np.mean((scatter_ratio_gbr >= 0.5) & (scatter_ratio_gbr <= 2.0)) * 100
within_1_5x_gbr = np.mean((scatter_ratio_gbr >= 0.67) & (scatter_ratio_gbr <= 1.5)) * 100
within_3x_gbr = np.mean((scatter_ratio_gbr >= 0.33) & (scatter_ratio_gbr <= 3.0)) * 100

# Conservative vs Optimistic predictions
conservative_gbr = np.mean(scatter_ratio_gbr < 1.0) * 100  # Under-predictions
optimistic_gbr = np.mean(scatter_ratio_gbr > 1.0) * 100    # Over-predictions

print(f"📊 GBR SCATTER BAND ANALYSIS:")
print(f"   ±2 Scatter Band:  {within_2x_gbr:.1f}% of predictions")
print(f"   ±1.5 Scatter Band: {within_1_5x_gbr:.1f}% of predictions") 
print(f"   ±3 Scatter Band:  {within_3x_gbr:.1f}% of predictions")
print(f"   Conservative predictions (<1.0): {conservative_gbr:.1f}%")
print(f"   Optimistic predictions (>1.0): {optimistic_gbr:.1f}%")

# ============================================================
# SCATTER BAND PLOT
# ============================================================
plt.figure(figsize=(10, 8))

# Scatter plot
plt.scatter(y_test_actual, y_pred_test_gbr, alpha=0.7, color='blue', s=50, 
           label=f'GBR Predictions ({within_2x_gbr:.1f}% within ±2 band)')

# Reference lines
max_val = max(y_test_actual.max(), y_pred_test_gbr.max())
min_val = min(y_test_actual.min(), y_pred_test_gbr.min())

# Perfect prediction line
plt.plot([min_val, max_val], [min_val, max_val], 'k-', label='Perfect Prediction', linewidth=2)

# ±2 scatter band lines
plt.plot([min_val, max_val], [0.5*min_val, 0.5*max_val], 'r--', label='±2 Scatter Band', linewidth=1.5)
plt.plot([min_val, max_val], [2*min_val, 2*max_val], 'r--', linewidth=1.5)

plt.xscale('log')
plt.yscale('log')
plt.xlabel('Actual Rupture Time (hours)')
plt.ylabel('Predicted Rupture Time (hours)')
plt.title('GBR: ±2 Scatter Band Analysis\n(Actual Scale)')
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

# ============================================================
# SCATTER RATIO DISTRIBUTION (IMPROVED VERSION)
# ============================================================
plt.figure(figsize=(10, 6))

# Use consistent binning and add statistics
n, bins, patches = plt.hist(scatter_ratio_gbr, bins=50, alpha=0.7, color='blue', 
                           edgecolor='black', density=False)

plt.axvline(1.0, color='red', linestyle='--', linewidth=2, label='Perfect Prediction (1.0)')
plt.axvline(0.5, color='orange', linestyle='--', linewidth=1.5, label='±2 Band Limits')
plt.axvline(2.0, color='orange', linestyle='--', linewidth=1.5)

# Add statistical annotations
median_ratio = np.median(scatter_ratio_gbr)
mean_ratio = np.mean(scatter_ratio_gbr)

plt.axvline(median_ratio, color='green', linestyle=':', linewidth=2, 
           label=f'Median Ratio: {median_ratio:.2f}')

plt.xlabel('Predicted / Actual Ratio')
plt.ylabel('Frequency')
plt.title(f'GBR: Scatter Ratio Distribution\n'
          f'Mean: {mean_ratio:.2f}, Median: {median_ratio:.2f}, '
          f'Within ±2: {within_2x_gbr:.1f}%')
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()

# Add text box with key statistics
textstr = f'''Key Statistics:
N = {len(scatter_ratio_gbr)}
Mean = {mean_ratio:.2f}
Median = {median_ratio:.2f}
Std = {np.std(scatter_ratio_gbr):.2f}
Within ±2: {within_2x_gbr:.1f}%'''
props = dict(boxstyle='round', facecolor='wheat', alpha=0.8)
plt.gca().text(0.95, 0.95, textstr, transform=plt.gca().transAxes, 
               verticalalignment='top', horizontalalignment='right', bbox=props)

plt.show()

### XGBOOST

In [ ]:
from xgboost import XGBRegressor
from sklearn.model_selection import GridSearchCV, KFold, cross_val_predict
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler  
from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error
import numpy as np
import pandas as pd

# ============================================================
# 🔄 XGBOOST PIPELINE (LOG TARGET, LOG-SCALE METRICS)
# ============================================================
print("🔄 Performing XGBoost hyperparameter tuning on LOG target...")

pipeline_xgb = Pipeline([
    ('scaler', StandardScaler()),
    ('xgb', XGBRegressor(
        objective='reg:squarederror',
        random_state=42,
        n_jobs=-1
    ))
])

param_grid_xgb = {
    'xgb__n_estimators': [100, 200, 300],
    'xgb__max_depth': [3, 5, 7],
    'xgb__learning_rate': [0.01, 0.05, 0.1],
    'xgb__subsample': [0.8, 0.9, 1.0],
    'xgb__colsample_bytree': [0.8, 1.0]
}

kf = KFold(n_splits=5, shuffle=True, random_state=42)

grid_search_xgb = GridSearchCV(
    pipeline_xgb,
    param_grid_xgb,
    cv=kf,
    scoring='r2',
    n_jobs=-1,
    verbose=1,
    refit=True
)

# ------------------------------------------------------------
# FIT GRID SEARCH
# ------------------------------------------------------------
grid_search_xgb.fit(X_train, y_train)
best_xgb_pipeline = grid_search_xgb.best_estimator_

print(f"🎯 Best XGBoost Parameters: {grid_search_xgb.best_params_}")
print(f"🎯 Best CV R² (log scale): {grid_search_xgb.best_score_:.4f}")

# SHOW TOP PARAMETER COMBINATIONS (This is optional) 
# ------------------------------------------------------------
results_df_xgb = pd.DataFrame(grid_search_xgb.cv_results_)
print(f"\n🔍 Top 5 XGBoost parameter combinations:")
print(results_df_xgb[['params', 'mean_test_score']].sort_values('mean_test_score', ascending=False).head(5))
# ------------------------------------------------------------

# ============================================================
# PREDICTIONS (LOG SCALE)
# ============================================================
y_pred_log_train_xgb = best_xgb_pipeline.predict(X_train)
y_pred_log_test_xgb = best_xgb_pipeline.predict(X_test)

# ============================================================
# METRICS FUNCTION (LOG SCALE)
# ============================================================
def compute_metrics(y_true, y_pred):
    r2_xgb = r2_score(y_true, y_pred)
    mse_xgb = mean_squared_error(y_true, y_pred)
    mae_xgb = mean_absolute_error(y_true, y_pred)
    return r2_xgb, mse_xgb, mae_xgb

# Training & Testing metrics (log scale)
r2_xgb_train, mse_xgb_train, mae_xgb_train = compute_metrics(y_train, y_pred_log_train_xgb)
r2_xgb_test, mse_xgb_test, mae_xgb_test = compute_metrics(y_test, y_pred_log_test_xgb)

# ------------------------------------------------------------
# CROSS-VALIDATION R² (LOG SCALE)
# ------------------------------------------------------------
y_cv_pred_log_xgb = cross_val_predict(best_xgb_pipeline, X_train, y_train, cv=kf)
cv_r2_xgb = r2_score(y_train, y_cv_pred_log_xgb)

# ============================================================
# DISPLAY RESULTS (LOG SCALE)
# ============================================================
print("\n📊 XGBOOST PERFORMANCE (LOG SCALE)")
print("="*60)
print(f"Training R² (log): {r2_xgb_train:.4f}")
print(f"Testing  R² (log): {r2_xgb_test:.4f}")
print(f"Cross-Validation R² (log): {cv_r2_xgb:.4f}\n")

print(f"Training MSE (log): {mse_xgb_train:.4f}")
print(f"Testing  MSE (log): {mse_xgb_test:.4f}")
print(f"Training MAE (log): {mae_xgb_train:.4f}")
print(f"Testing  MAE (log): {mae_xgb_test:.4f}")

print("\nR² Difference (Train - Test):", round(r2_xgb_train - r2_xgb_test, 4))
print("Overfitting Indicator:", "⚠️ HIGH (Overfitting)" if (r2_xgb_train - r2_xgb_test) > 0.2 else "✅ GOOD")

# ------------------------------------------------------------
# PREDICTIONS IN BOTH LOG AND ACTUAL SCALE
# ------------------------------------------------------------
y_pred_train_xgb = np.exp(y_pred_log_train_xgb)
y_pred_test_xgb = np.exp(y_pred_log_test_xgb)
y_train_actual = np.exp(y_train)
y_test_actual = np.exp(y_test)

# First 10 predictions in LOG SCALE
print("\nFirst 10 Predictions - Test Set (LOG SCALE)")
comparison_log_df_xgb = pd.DataFrame({
    "Actual (log Rupture Time)": y_test[:10],
    "Predicted (log)": y_pred_log_test_xgb[:10],
    "Error": y_test[:10] - y_pred_log_test_xgb[:10]
}).round(4)
print(comparison_log_df_xgb)

# First 10 predictions in ACTUAL SCALE
print("\nFirst 10 Predictions - Test Set (ACTUAL SCALE)")
comparison_actual_df_xgb = pd.DataFrame({
    "Actual (Rupture Time)": y_test_actual[:10],
    "Predicted": y_pred_test_xgb[:10],
    "Error": y_test_actual[:10] - y_pred_test_xgb[:10]
}).round(2)
print(comparison_actual_df_xgb)

### XGBOOST Plots

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
import pandas as pd

# Residuals
train_residuals_xgb = y_train - y_pred_log_train_xgb
test_residuals_xgb = y_test - y_pred_log_test_xgb

# -----------------------------
# SET PLOT STYLE
# -----------------------------
sns.set_style("whitegrid")
plt.rcParams.update({
    'figure.figsize': (7, 5),
    'axes.titlesize': 16,
    'axes.labelsize': 14,
    'xtick.labelsize': 12,
    'ytick.labelsize': 12,
    'lines.linewidth': 2,
    'lines.markersize': 6
})

# ============================================================
# 1️⃣ Predicted vs Actual (Log Scale)
# ============================================================
plt.figure()
sns.scatterplot(x=y_train, y=y_pred_log_train_xgb, label='Train', color='steelblue')
sns.scatterplot(x=y_test, y=y_pred_log_test_xgb, label='Test', color='orange')

# Perfect prediction line
lims = [min(y_train.min(), y_test.min()), max(y_train.max(), y_test.max())]
plt.plot(lims, lims, 'k--', alpha=0.7)
plt.xlabel("Actual log(Rupture Time)")
plt.ylabel("Predicted log(Rupture Time)")
plt.title("XGBOOST: Predicted vs Actual (Log Scale)")
plt.legend()
plt.tight_layout()
plt.show()

# ============================================================
# 2️⃣ Residuals Plot (Log Scale)
# ============================================================
plt.figure()
sns.scatterplot(x=y_train, y=train_residuals_xgb, label='Train', color='steelblue')
sns.scatterplot(x=y_test, y=test_residuals_xgb, label='Test', color='orange')
plt.axhline(0, color='k', linestyle='--', alpha=0.7)
plt.xlabel("Actual log(Rupture Time)")
plt.ylabel("Residuals")
plt.title("XGBOOST: Residuals vs Actual (Log Scale)")
plt.legend()
plt.tight_layout()
plt.show()

# ============================================================
# 3️⃣ Histogram of Errors (Log Scale)
# ============================================================
plt.figure()
sns.histplot(train_residuals_xgb, color='steelblue', label='Train', kde=True, bins=30)
sns.histplot(test_residuals_xgb, color='orange', label='Test', kde=True, bins=30)
plt.xlabel("Residuals")
plt.ylabel("Count")
plt.title("XGBOOST: Histogram of Residuals (Log Scale)")
plt.legend()
plt.tight_layout()
plt.show()


### XGBOOST ±2 Scatter Band Analysis

In [ ]:
# ============================================================
# 🔬 ±2 SCATTER BAND ANALYSIS: XGBOOST
# ============================================================
print("🔬 ANALYZING ±2 SCATTER BAND FOR XGBOOST...")
print("="*60)

# Calculate scatter ratios (Predicted/Actual)
scatter_ratio_xgb = y_pred_test_xgb / y_test_actual

# Scatter band metrics (FIXED variable names - no ± symbol)
within_2x_xgb = np.mean((scatter_ratio_xgb >= 0.5) & (scatter_ratio_xgb <= 2.0)) * 100
within_1_5x_xgb = np.mean((scatter_ratio_xgb >= 0.67) & (scatter_ratio_xgb <= 1.5)) * 100
within_3x_xgb = np.mean((scatter_ratio_xgb >= 0.33) & (scatter_ratio_xgb <= 3.0)) * 100

# Conservative vs Optimistic predictions
conservative_xgb = np.mean(scatter_ratio_xgb < 1.0) * 100  # Under-predictions
optimistic_xgb = np.mean(scatter_ratio_xgb > 1.0) * 100    # Over-predictions

print(f"📊 XGBOOST SCATTER BAND ANALYSIS:")
print(f"   ±2 Scatter Band:  {within_2x_xgb:.1f}% of predictions")
print(f"   ±1.5 Scatter Band: {within_1_5x_xgb:.1f}% of predictions") 
print(f"   ±3 Scatter Band:  {within_3x_xgb:.1f}% of predictions")
print(f"   Conservative predictions (<1.0): {conservative_xgb:.1f}%")
print(f"   Optimistic predictions (>1.0): {optimistic_xgb:.1f}%")

# ============================================================
# SCATTER BAND PLOT
# ============================================================
plt.figure(figsize=(10, 8))

# Scatter plot
plt.scatter(y_test_actual, y_pred_test_xgb, alpha=0.7, color='blue', s=50, 
           label=f'XGBOOST Predictions ({within_2x_xgb:.1f}% within ±2 band)')

# Reference lines
max_val = max(y_test_actual.max(), y_pred_test_xgb.max())
min_val = min(y_test_actual.min(), y_pred_test_xgb.min())

# Perfect prediction line
plt.plot([min_val, max_val], [min_val, max_val], 'k-', label='Perfect Prediction', linewidth=2)

# ±2 scatter band lines
plt.plot([min_val, max_val], [0.5*min_val, 0.5*max_val], 'r--', label='±2 Scatter Band', linewidth=1.5)
plt.plot([min_val, max_val], [2*min_val, 2*max_val], 'r--', linewidth=1.5)

plt.xscale('log')
plt.yscale('log')
plt.xlabel('Actual Rupture Time (hours)')
plt.ylabel('Predicted Rupture Time (hours)')
plt.title('XGBOOST: ±2 Scatter Band Analysis\n(Actual Scale)')
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

# ============================================================
# SCATTER RATIO DISTRIBUTION (IMPROVED VERSION)
# ============================================================
plt.figure(figsize=(10, 6))

# Use consistent binning and add statistics
n, bins, patches = plt.hist(scatter_ratio_xgb, bins=50, alpha=0.7, color='blue', 
                           edgecolor='black', density=False)

plt.axvline(1.0, color='red', linestyle='--', linewidth=2, label='Perfect Prediction (1.0)')
plt.axvline(0.5, color='orange', linestyle='--', linewidth=1.5, label='±2 Band Limits')
plt.axvline(2.0, color='orange', linestyle='--', linewidth=1.5)

# Add statistical annotations
median_ratio = np.median(scatter_ratio_xgb)
mean_ratio = np.mean(scatter_ratio_xgb)

plt.axvline(median_ratio, color='green', linestyle=':', linewidth=2, 
           label=f'Median Ratio: {median_ratio:.2f}')

plt.xlabel('Predicted / Actual Ratio')
plt.ylabel('Frequency')
plt.title(f'XGBOOST: Scatter Ratio Distribution\n'
          f'Mean: {mean_ratio:.2f}, Median: {median_ratio:.2f}, '
          f'Within ±2: {within_2x_xgb:.1f}%')
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()

# Add text box with key statistics
textstr = f'''Key Statistics:
N = {len(scatter_ratio_xgb)}
Mean = {mean_ratio:.2f}
Median = {median_ratio:.2f}
Std = {np.std(scatter_ratio_xgb):.2f}
Within ±2: {within_2x_xgb:.1f}%'''
props = dict(boxstyle='round', facecolor='wheat', alpha=0.8)
plt.gca().text(0.95, 0.95, textstr, transform=plt.gca().transAxes, 
               verticalalignment='top', horizontalalignment='right', bbox=props)

plt.show()

### DETAILED COMPARISON PLOTS FOR ALL MODELS

In [ ]:
# ============================================================
# 📊 HIGH-QUALITY COMBINED PLOTS - ALL 5 ML MODELS (FULL HD)
# ============================================================
print("📊 CREATING HIGH-QUALITY COMBINED PLOTS FOR ALL ML MODELS...")
print("="*70)

import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
import pandas as pd
from sklearn.metrics import r2_score

# ============================================================
# SET HIGH-QUALITY PLOT STYLE
# ============================================================
plt.style.use('default')
sns.set_style("whitegrid")
plt.rcParams.update({
    'font.size': 12,
    'font.family': 'DejaVu Sans',
    'figure.dpi': 300,
    'savefig.dpi': 300,
    'figure.figsize': (20, 12),
    'axes.titlesize': 16,
    'axes.labelsize': 14,
    'xtick.labelsize': 12,
    'ytick.labelsize': 12,
    'legend.fontsize': 11,
    'legend.frameon': True,
    'legend.framealpha': 0.9,
    'lines.linewidth': 2.5,
    'lines.markersize': 8,
    'axes.linewidth': 1.2,
    'grid.alpha': 0.3
})

# ============================================================
# CREATE COMPREHENSIVE DATA DICTIONARY
# ============================================================
models_data = {
    'Linear Regression': {
        'y_pred_train': y_pred_log_train_lr,
        'y_pred_test': y_pred_log_test_lr,
        'train_residuals': train_residuals_lr,
        'test_residuals': test_residuals_lr,
        'color_train': '#1f77b4',  # Better color scheme
        'color_test': '#ff7f0e',
        'marker_train': 'o',
        'marker_test': 's'
    },
    'SVR': {
        'y_pred_train': y_pred_log_train_svr,
        'y_pred_test': y_pred_log_test_svr,
        'train_residuals': train_residuals_svr,
        'test_residuals': test_residuals_svr,
        'color_train': '#2ca02c',
        'color_test': '#d62728',
        'marker_train': 'o',
        'marker_test': 's'
    },
    'Random Forest': {
        'y_pred_train': y_pred_log_train_rf,
        'y_pred_test': y_pred_log_test_rf,
        'train_residuals': train_residuals_rf,
        'test_residuals': test_residuals_rf,
        'color_train': '#9467bd',
        'color_test': '#8c564b',
        'marker_train': 'o',
        'marker_test': 's'
    },
    'Gradient Boosting': {
        'y_pred_train': y_pred_log_train_gbr,
        'y_pred_test': y_pred_log_test_gbr,
        'train_residuals': train_residuals_gbr,
        'test_residuals': test_residuals_gbr,
        'color_train': '#e377c2',
        'color_test': '#7f7f7f',
        'marker_train': 'o',
        'marker_test': 's'
    },
    'XGBoost': {
        'y_pred_train': y_pred_log_train_xgb,
        'y_pred_test': y_pred_log_test_xgb,
        'train_residuals': train_residuals_xgb,
        'test_residuals': test_residuals_xgb,
        'color_train': '#bcbd22',
        'color_test': '#17becf',
        'marker_train': 'o',
        'marker_test': 's'
    }
}

# ============================================================
# PLOT 1: PREDICTED VS ACTUAL (LOG SCALE) - 5 SUBPLOTS
# ============================================================
print("🔄 Creating Plot 1: Predicted vs Actual...")

fig1, axes1 = plt.subplots(2, 3, figsize=(24, 16))
axes1 = axes1.ravel()
fig1.delaxes(axes1[5])  # Remove empty subplot

for i, (model_name, model_data) in enumerate(models_data.items()):
    ax = axes1[i]
    
    # Calculate R² scores
    r2_train = r2_score(y_train, model_data['y_pred_train'])
    r2_test = r2_score(y_test, model_data['y_pred_test'])
    
    # Scatter plots with enhanced styling
    ax.scatter(y_train, model_data['y_pred_train'], 
               alpha=0.8, color=model_data['color_train'], 
               marker=model_data['marker_train'], s=80,
               label=f'Train (R² = {r2_train:.4f})', edgecolor='white', linewidth=0.5)
    
    ax.scatter(y_test, model_data['y_pred_test'], 
               alpha=0.8, color=model_data['color_test'], 
               marker=model_data['marker_test'], s=80,
               label=f'Test (R² = {r2_test:.4f})', edgecolor='white', linewidth=0.5)
    
    # Perfect prediction line
    lims = [min(y_train.min(), y_test.min()), max(y_train.max(), y_test.max())]
    ax.plot(lims, lims, 'k-', alpha=0.8, linewidth=3, label='Perfect Prediction')
    
    # Enhanced styling
    ax.set_xlabel("Actual log(Rupture Time)", fontsize=14, fontweight='bold')
    ax.set_ylabel("Predicted log(Rupture Time)", fontsize=14, fontweight='bold')
    ax.set_title(f"{model_name}\nPredicted vs Actual", fontsize=16, fontweight='bold', pad=20)
    
    # Add R² info to plot
    ax.text(0.05, 0.95, f'Test R² = {r2_test:.4f}', transform=ax.transAxes,
            fontsize=12, fontweight='bold', verticalalignment='top',
            bbox=dict(boxstyle='round', facecolor='lightgray', alpha=0.8))
    
    ax.legend(loc='lower right', framealpha=0.95, fancybox=True)
    ax.grid(True, alpha=0.4)
    
    # Set equal aspect ratio for better visualization
    ax.set_aspect('equal')

# Add overall title
fig1.suptitle('Machine Learning Models: Predicted vs Actual (Log Scale)', 
              fontsize=20, fontweight='bold', y=0.99)

plt.tight_layout()
plt.subplots_adjust(top=0.92)

# Save in Full HD
plt.savefig('ML_Models_Predicted_vs_Actual_FullHD.png', dpi=300, bbox_inches='tight', 
            facecolor='white', edgecolor='none')
print("✅ Plot 1 saved as 'ML_Models_Predicted_vs_Actual_FullHD.png'")
plt.show()

# ============================================================
# PLOT 2: RESIDUALS VS ACTUAL (LOG SCALE) - 5 SUBPLOTS
# ============================================================
print("🔄 Creating Plot 2: Residuals vs Actual...")

fig2, axes2 = plt.subplots(2, 3, figsize=(24, 16))
axes2 = axes2.ravel()
fig2.delaxes(axes2[5])  # Remove empty subplot

for i, (model_name, model_data) in enumerate(models_data.items()):
    ax = axes2[i]
    
    # Calculate residual statistics
    train_std = np.std(model_data['train_residuals'])
    test_std = np.std(model_data['test_residuals'])
    
    # Residuals scatter plots
    ax.scatter(y_train, model_data['train_residuals'], 
               alpha=0.7, color=model_data['color_train'], 
               marker=model_data['marker_train'], s=70,
               label=f'Train Residuals (σ = {train_std:.3f})', edgecolor='white', linewidth=0.5)
    
    ax.scatter(y_test, model_data['test_residuals'], 
               alpha=0.7, color=model_data['color_test'], 
               marker=model_data['marker_test'], s=70,
               label=f'Test Residuals (σ = {test_std:.3f})', edgecolor='white', linewidth=0.5)
    
    # Zero error line
    ax.axhline(y=0, color='red', linestyle='--', alpha=0.8, linewidth=3, label='Zero Error')
    
    # Enhanced styling
    ax.set_xlabel("Actual log(Rupture Time)", fontsize=14, fontweight='bold')
    ax.set_ylabel("Residuals", fontsize=14, fontweight='bold')
    ax.set_title(f"{model_name}\nResiduals vs Actual", fontsize=16, fontweight='bold', pad=20)
    
    ax.legend(loc='best', framealpha=0.95, fancybox=True)
    ax.grid(True, alpha=0.4)

# Add overall title
fig2.suptitle('Machine Learning Models: Residuals vs Actual (Log Scale)', 
              fontsize=20, fontweight='bold', y=0.99)

plt.tight_layout()
plt.subplots_adjust(top=0.92)

# Save in Full HD
plt.savefig('ML_Models_Residuals_vs_Actual_FullHD.png', dpi=300, bbox_inches='tight', 
            facecolor='white', edgecolor='none')
print("✅ Plot 2 saved as 'ML_Models_Residuals_vs_Actual_FullHD.png'")
plt.show()

# ============================================================
# PLOT 3: RESIDUALS HISTOGRAMS - 5 SUBPLOTS
# ============================================================
print("🔄 Creating Plot 3: Residuals Histograms...")

fig3, axes3 = plt.subplots(2, 3, figsize=(24, 16))
axes3 = axes3.ravel()
fig3.delaxes(axes3[5])  # Remove empty subplot

for i, (model_name, model_data) in enumerate(models_data.items()):
    ax = axes3[i]
    
    # Calculate statistics for annotations
    train_mean = np.mean(model_data['train_residuals'])
    test_mean = np.mean(model_data['test_residuals'])
    train_std = np.std(model_data['train_residuals'])
    test_std = np.std(model_data['test_residuals'])
    
    # Enhanced histograms with KDE
    sns.histplot(model_data['train_residuals'], color=model_data['color_train'], 
                 label=f'Train (μ={train_mean:.3f}, σ={train_std:.3f})', 
                 kde=True, bins=25, alpha=0.7, ax=ax, edgecolor='black', linewidth=0.5)
    
    sns.histplot(model_data['test_residuals'], color=model_data['color_test'], 
                 label=f'Test (μ={test_mean:.3f}, σ={test_std:.3f})', 
                 kde=True, bins=25, alpha=0.7, ax=ax, edgecolor='black', linewidth=0.5)
    
    # Zero line
    ax.axvline(x=0, color='red', linestyle='--', alpha=0.8, linewidth=3, label='Zero Error')
    
    # Enhanced styling
    ax.set_xlabel("Residuals", fontsize=14, fontweight='bold')
    ax.set_ylabel("Density", fontsize=14, fontweight='bold')
    ax.set_title(f"{model_name}\nResiduals Distribution", fontsize=16, fontweight='bold', pad=20)
    
    ax.legend(loc='best', framealpha=0.95, fancybox=True)
    ax.grid(True, alpha=0.4)

# Add overall title
fig3.suptitle('Machine Learning Models: Residuals Distribution (Log Scale)', 
              fontsize=20, fontweight='bold', y=0.99)

plt.tight_layout()
plt.subplots_adjust(top=0.92)

# Save in Full HD
plt.savefig('ML_Models_Residuals_Histograms_FullHD.png', dpi=300, bbox_inches='tight', 
            facecolor='white', edgecolor='none')
print("✅ Plot 3 saved as 'ML_Models_Residuals_Histograms_FullHD.png'")
plt.show()

# ============================================================
# COMPREHENSIVE PERFORMANCE SUMMARY
# ============================================================
print("\n📈 COMPREHENSIVE PERFORMANCE SUMMARY")
print("="*60)

performance_data = []
for model_name, model_data in models_data.items():
    # Calculate all metrics
    r2_train = r2_score(y_train, model_data['y_pred_train'])
    r2_test = r2_score(y_test, model_data['y_pred_test'])
    train_std = np.std(model_data['train_residuals'])
    test_std = np.std(model_data['test_residuals'])
    train_mean_res = np.mean(model_data['train_residuals'])
    test_mean_res = np.mean(model_data['test_residuals'])
    
    performance_data.append({
        'Model': model_name,
        'Train R²': r2_train,
        'Test R²': r2_test,
        'R² Difference': r2_train - r2_test,
        'Train Residual Std': train_std,
        'Test Residual Std': test_std,
        'Train Residual Mean': train_mean_res,
        'Test Residual Mean': test_mean_res,
        'Overfitting': '⚠️ HIGH' if (r2_train - r2_test) > 0.2 else '✅ GOOD'
    })

performance_df = pd.DataFrame(performance_data).round(4)
print("\n📊 Detailed Performance Metrics:")
print(performance_df.to_string(index=False))

# Find best models
best_test_r2 = performance_df.loc[performance_df['Test R²'].idxmax()]
best_residual_std = performance_df.loc[performance_df['Test Residual Std'].idxmin()]

print(f"\n🏆 BEST MODEL BY TEST R²: {best_test_r2['Model']} (R² = {best_test_r2['Test R²']:.4f})")
print(f"🏆 BEST MODEL BY RESIDUAL STD: {best_residual_std['Model']} (σ = {best_residual_std['Test Residual Std']:.4f})")

print(f"\n💾 All plots saved in Full HD (1920x1080) resolution!")
print("   - ML_Models_Predicted_vs_Actual_FullHD.png")
print("   - ML_Models_Residuals_vs_Actual_FullHD.png") 
print("   - ML_Models_Residuals_Histograms_FullHD.png")

### DETAILED ±2 SCATTER BAND COMPARISON PLOTS FOR ALL MODELS

In [ ]:
# ============================================================
# 📊 COMPREHENSIVE COMPARISON PLOTS - ALL 5 MODELS (DETAILED)
# ============================================================
print("📊 CREATING DETAILED COMPARISON PLOTS FOR ALL MODELS...")
print("="*60)

# Create a dictionary with all model results for easy access
models_data = {
    'Linear Regression': {
        'y_pred_actual': y_pred_test_lr,
        'scatter_ratio': scatter_ratio_lr,
        'within_2x': within_2x_lr,
        'color': 'blue'
    },
    'SVR': {
        'y_pred_actual': y_pred_test_svr,
        'scatter_ratio': scatter_ratio_svr,
        'within_2x': within_2x_svr,
        'color': 'red'
    },
    'Random Forest': {
        'y_pred_actual': y_pred_test_rf,
        'scatter_ratio': scatter_ratio_rf,
        'within_2x': within_2x_rf,
        'color': 'green'
    },
    'Gradient Boosting': {
        'y_pred_actual': y_pred_test_gbr,
        'scatter_ratio': scatter_ratio_gbr,
        'within_2x': within_2x_gbr,
        'color': 'orange'
    },
    'XGBoost': {
        'y_pred_actual': y_pred_test_xgb,
        'scatter_ratio': scatter_ratio_xgb,
        'within_2x': within_2x_xgb,
        'color': 'purple'
    }
}

# ============================================================
# PLOT 1: DETAILED SCATTER BAND COMPARISON (5 SUBPLOTS)
# ============================================================
fig1, axes1 = plt.subplots(2, 3, figsize=(20, 12))  # 2 rows, 3 columns
axes1 = axes1.ravel()  # Flatten for easy indexing

# Remove the last subplot (we only need 5)
fig1.delaxes(axes1[5])

model_names = list(models_data.keys())

for i, (model_name, model_data) in enumerate(models_data.items()):
    ax = axes1[i]
    y_pred_actual = model_data['y_pred_actual']
    within_2x = model_data['within_2x']
    
    # Scatter plot with detailed styling
    ax.scatter(y_test_actual, y_pred_actual, alpha=0.7, color=model_data['color'], s=50, 
               label=f'{model_name} ({within_2x:.1f}% within ±2)')
    
    # Reference lines
    max_val = max(y_test_actual.max(), y_pred_actual.max())
    min_val = min(y_test_actual.min(), y_pred_actual.min())
    
    # Perfect prediction line
    ax.plot([min_val, max_val], [min_val, max_val], 'k-', label='Perfect Prediction', linewidth=2)
    
    # ±2 scatter band lines
    ax.plot([min_val, max_val], [0.5*min_val, 0.5*max_val], 'r--', label='±2 Scatter Band', linewidth=1.5)
    ax.plot([min_val, max_val], [2*min_val, 2*max_val], 'r--', linewidth=1.5)
    
    # Set scales and labels
    ax.set_xscale('log')
    ax.set_yscale('log')
    ax.set_xlabel('Actual Rupture Time (hours)', fontsize=11)
    ax.set_ylabel('Predicted Rupture Time (hours)', fontsize=11)
    ax.set_title(f'{model_name}: ±2 Scatter Band Analysis\n(Actual Scale)', fontsize=12, fontweight='bold')
    
    # Add legend and grid
    ax.legend(fontsize=9, loc='upper left')
    ax.grid(True, alpha=0.3)
    
    # Set consistent limits for better comparison
    ax.set_xlim(min_val * 0.8, max_val * 1.2)
    ax.set_ylim(min_val * 0.8, max_val * 1.2)

plt.tight_layout()
plt.show()

# ============================================================
# PLOT 2: DETAILED SCATTER RATIO DISTRIBUTION (5 SUBPLOTS)
# ============================================================
fig2, axes2 = plt.subplots(2, 3, figsize=(20, 12))  # 2 rows, 3 columns
axes2 = axes2.ravel()  # Flatten for easy indexing

# Remove the last subplot (we only need 5)
fig2.delaxes(axes2[5])

for i, (model_name, model_data) in enumerate(models_data.items()):
    ax = axes2[i]
    scatter_ratio = model_data['scatter_ratio']
    within_2x = model_data['within_2x']
    
    # Calculate statistics for each model
    median_ratio = np.median(scatter_ratio)
    mean_ratio = np.mean(scatter_ratio)
    std_ratio = np.std(scatter_ratio)
    
    # Detailed histogram with consistent binning
    n, bins, patches = ax.hist(scatter_ratio, bins=50, alpha=0.7, color=model_data['color'], 
                              edgecolor='black', density=False)
    
    # Reference lines with detailed labels
    ax.axvline(1.0, color='red', linestyle='--', linewidth=2, label='Perfect Prediction (1.0)')
    ax.axvline(0.5, color='orange', linestyle='--', linewidth=1.5, label='±2 Band Limits')
    ax.axvline(2.0, color='orange', linestyle='--', linewidth=1.5)
    ax.axvline(median_ratio, color='green', linestyle=':', linewidth=2, 
               label=f'Median Ratio: {median_ratio:.2f}')
    
    # Labels and title
    ax.set_xlabel('Predicted / Actual Ratio', fontsize=11)
    ax.set_ylabel('Frequency', fontsize=11)
    ax.set_title(f'{model_name}: Scatter Ratio Distribution\n'
                f'Mean: {mean_ratio:.2f}, Median: {median_ratio:.2f}, '
                f'Within ±2: {within_2x:.1f}%', fontsize=12, fontweight='bold')
    
    # Legend and grid
    ax.legend(fontsize=9)
    ax.grid(True, alpha=0.3)
    
    # Add detailed statistics box (like original)
    textstr = f'''Key Statistics:
N = {len(scatter_ratio)}
Mean = {mean_ratio:.2f}
Median = {median_ratio:.2f}
Std = {std_ratio:.2f}
Within ±2: {within_2x:.1f}%'''
    
    props = dict(boxstyle='round', facecolor='wheat', alpha=0.8)
    ax.text(0.95, 0.95, textstr, transform=ax.transAxes, fontsize=9,
            verticalalignment='top', horizontalalignment='right', bbox=props)

plt.tight_layout()
plt.show()

# ============================================================
# DETAILED SUMMARY TABLE WITH ALL METRICS (CORRECTED)
# ============================================================
print("\n📈 DETAILED SUMMARY OF ALL MODELS")
print("="*80)

detailed_summary = []
# FIXED: Remove enumerate() - we need the actual model name and data
for model_name, model_data in models_data.items():  # ← FIXED THIS LINE
    scatter_ratio = model_data['scatter_ratio']
    
    # Calculate all metrics for each model
    within_1_5x = np.mean((scatter_ratio >= 0.67) & (scatter_ratio <= 1.5)) * 100
    within_3x = np.mean((scatter_ratio >= 0.33) & (scatter_ratio <= 3.0)) * 100
    conservative = np.mean(scatter_ratio < 1.0) * 100
    optimistic = np.mean(scatter_ratio > 1.0) * 100
    
    detailed_summary.append({
        'Model': model_name,  # ← Now this will be the actual model name
        '±2 Band (%)': model_data['within_2x'],
        '±1.5 Band (%)': within_1_5x,
        '±3 Band (%)': within_3x,
        'Mean Ratio': np.mean(scatter_ratio),
        'Median Ratio': np.median(scatter_ratio),
        'Std Ratio': np.std(scatter_ratio),
        'Conservative (%)': conservative,
        'Optimistic (%)': optimistic
    })

# Create and display detailed summary
detailed_df = pd.DataFrame(detailed_summary)
detailed_df = detailed_df.round(2)

print("\n📊 PERFORMANCE METRICS COMPARISON:")
print(detailed_df.to_string(index=False))

# Rank models by ±2 scatter band performance
ranked_df = detailed_df.sort_values('±2 Band (%)', ascending=False)
print(f"\n🏆 MODEL RANKING BY ±2 SCATTER BAND PERFORMANCE:")
for i, (idx, row) in enumerate(ranked_df.iterrows()):
    print(f"  {i+1}. {row['Model']}: {row['±2 Band (%)']}% within ±2 band")

best_model = ranked_df.iloc[0]
print(f"\n🎯 BEST PERFORMER: {best_model['Model']}")
print(f"   - {best_model['±2 Band (%)']}% within ±2 scatter band")
print(f"   - {best_model['±1.5 Band (%)']}% within ±1.5 scatter band")
print(f"   - Mean Ratio: {best_model['Mean Ratio']:.2f}, Median Ratio: {best_model['Median Ratio']:.2f}")